In [1]:
import numpy as np
import tarfile
import os
import glob
import sys

print(f"Numpy version: {np.__version__}")

Numpy version: 1.26.4


In [2]:
# --- 設定項目 ---

# 1. ダウンロードした .tar.gz ファイルへのパス
# (例: "PS_mocks/linear/Power_Spectrum_cmass_ngc_v5_Patchy.tar.gz")
TAR_FILE_PATH = "./boss dr12/Power_Spectrum_cmass_ngc_v5_Patchy.tar.gz"


# 2. 展開先のディレクトリ名
EXTRACT_DIR = "./patchy_mocks_cmass_ngc"

# 3. フィットに使う最大の k [h/Mpc] (README Section 5 推奨値)
K_MAX_FIT = 0.20

# 4. Cocoa形式で出力する共分散行列のファイル名
OUTPUT_COV_FILE = "kaiser_rsd_covariance.txt"

# 5. Cocoa形式で出力するデータベクトル（平均値）のファイル名
OUTPUT_MEAN_DATA_FILE = "kaiser_rsd_datavector_mean.txt"

In [3]:
print(f"'{TAR_FILE_PATH}' を '{EXTRACT_DIR}' に展開します...")
try:
    with tarfile.open(TAR_FILE_PATH, 'r:gz') as tar:
        tar.extractall(path=EXTRACT_DIR)
    print("展開完了。")
except FileNotFoundError:
    print(f"エラー: '{TAR_FILE_PATH}' が見つかりません。", file=sys.stderr)
except Exception as e:
    print(f"展開エラー: {e}", file=sys.stderr)

'./boss dr12/Power_Spectrum_cmass_ngc_v5_Patchy.tar.gz' を './patchy_mocks_cmass_ngc' に展開します...
展開完了。


In [4]:
print("モックファイルの読み込みを開始...")

# 展開したディレクトリ内の全テキストファイル（モック）を取得
# READMEのファイル名 'Power_Spectrum_cmass_ngc_v5_Patchy.tar.gz' に基づく
file_pattern = os.path.join(EXTRACT_DIR, "Power_Spectrum_cmass_ngc_v5_Patchy_*.txt")
mock_files = sorted(glob.glob(file_pattern))

if not mock_files:
    print(f"エラー: '{EXTRACT_DIR}' 内に '{file_pattern}' に一致するファイルが見つかりません。", file=sys.stderr)
    print("-> `EXTRACT_DIR` の中身を確認し、ファイルパスが正しいか確認してください。")
else:
    print(f"{len(mock_files)} 個のモックファイルを発見。")

    all_mock_vectors = []
    k_bins_filter_indices = None
    k_bins_count = 0

    for i, filepath in enumerate(mock_files):
        try:
            # README Section 3.1 形式:
            # 0:k-center, 1:k-eff, 2:P0, 3:P2, 4:P4, 5:N_modes, 6:P0_shotnoise
            # 使うのは 1:k-eff, 2:P0, 3:P2
            data = np.loadtxt(filepath, comments='#', usecols=(1, 2, 3))
        except Exception as e:
            print(f"読み込みエラー {filepath}: {e}", file=sys.stderr)
            continue
        
        # 最初のファイルで k<=K_MAX_FIT のインデックスを取得
        if i == 0:
            k_eff_values = data[:, 0]
            k_bins_filter_indices = np.where(k_eff_values <= K_MAX_FIT)[0]
            k_bins_count = len(k_bins_filter_indices)
            
            if k_bins_count == 0:
                print(f"エラー: k <= {K_MAX_FIT} のkビンがありません。", file=sys.stderr)
                break
            
            print(f"k <= {K_MAX_FIT} h/Mpc の {k_bins_count} ビンを使用します。")
            print(f"(データベクトルの全長は 2 * {k_bins_count} = {2 * k_bins_count} になります)")

        # k_max でフィルタリング
        p0 = data[k_bins_filter_indices, 1]
        p2 = data[k_bins_filter_indices, 2]
        
        # Cocoa 形式 [P0(k1), P2(k1), P0(k2), P2(k2), ...] に並べ替え
        interleaved_vector = np.empty(2 * k_bins_count)
        interleaved_vector[0::2] = p0  # 偶数インデックスに P0
        interleaved_vector[1::2] = p2  # 奇数インデックスに P2
        
        all_mock_vectors.append(interleaved_vector)

    # (N_real, N_bins) の形状を持つ Numpy 配列を作成
    if all_mock_vectors:
        mock_matrix = np.array(all_mock_vectors)
        print(f"モック行列の形状 (N_real, N_bins): {mock_matrix.shape}")
    else:
        print("データベクトルが作成されませんでした。")

モックファイルの読み込みを開始...
2048 個のモックファイルを発見。
k <= 0.2 h/Mpc の 20 ビンを使用します。
(データベクトルの全長は 2 * 20 = 40 になります)
モック行列の形状 (N_real, N_bins): (2048, 40)


In [5]:
if 'mock_matrix' in locals() and mock_matrix.size > 0:
    print("共分散行列を計算中...")
    
    # np.cov はデフォルトで (N-1) で割るため、Eq. 7 と一致します。
    # rowvar=False は、入力の形状が (N_real, N_bins) であることを示し、
    # (N_bins, N_bins) の共分散行列を返します。
    cov_matrix = np.cov(mock_matrix, rowvar=False)
    
    print(f"共分散行列の形状 (N_bins, N_bins): {cov_matrix.shape}")
    
    # 参考: モックの平均データベクトル (Eq. 8 の D_tilde)
    mean_data_vector = np.mean(mock_matrix, axis=0)
    print(f"平均データベクトルの形状 (N_bins,): {mean_data_vector.shape}")
    
else:
    print("モック行列が作成されていないため、計算をスキップします。")

共分散行列を計算中...
共分散行列の形状 (N_bins, N_bins): (40, 40)
平均データベクトルの形状 (N_bins,): (40,)
